In [1]:
import sys
sys.path.append("../src")

import csv
import torch
import torch.nn as nn
from data import get_loaders
from utils import set_seed

SEED = 42
set_seed(SEED)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

Using device: mps


In [15]:
#loads the Dogs and cats images with the data loader
#images are resized to 224 x 224 and normalised
#training images are randomly flipped, rotated and cropped
#loader hands images to the model in batches of 32
DATA_DIR = "../dataset"

train_loader, val_loader, test_loader = get_loaders(
    data_dir=DATA_DIR,
    batch_size=32,
    img_size=224,
    augment=True,
    train_subset=None,
    val_subset=None,
    seed=SEED,
)

images, labels = next(iter(train_loader))
print(images.shape)
print(labels[:10])

torch.Size([32, 3, 224, 224])
tensor([0, 0, 0, 0, 1, 1, 1, 0, 1, 0])


In [16]:
#builds the small CNN from scratch
#adaptiveAvgPool shrinks image sie to 4 x 4
class SmallCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128 * 4 * 4, 128), nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [17]:
EPOCHS = 5
rows = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}: train loss {avg_loss:.3f}, val accuracy {val_acc:.3f}")
    rows.append([epoch + 1, avg_loss, val_acc])

with open("../results/small_cnn_dogs_vs_cats.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_accuracy"])
    writer.writerows(rows)

Epoch 1: train loss 0.602, val accuracy 0.711
Epoch 2: train loss 0.513, val accuracy 0.779
Epoch 3: train loss 0.459, val accuracy 0.790
Epoch 4: train loss 0.422, val accuracy 0.795
Epoch 5: train loss 0.402, val accuracy 0.829
